In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
import time, os, json, random
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from thop import profile as thop_profile

# ============================================================
# Evaluation-only script — loads the ALREADY TRAINED BranchyNet
# weights produced by the HAM10000 BranchyNet training script
# (__ham10000__branchynet_resnet50.pth) and recomputes:
#   - final-exit accuracy / precision / recall / f1
#   - the early-exit threshold sweep (adaptive_forward)
#   - FLOPs, params, disk size, CPU/GPU inference timing
# No training happens here.
#
# IMPORTANT: this uses the BranchyResNet50 architecture (stem +
# branch1/branch2 + final fc). It is NOT compatible with plain
# ResNet-50 weights (e.g. __ham10000__baseline_resnet50.pth) —
# use ham10000_evaluate_saved_model.py for that one instead.
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Must match the training script exactly ─────────────────────
SEED        = 42
NUM_CLASSES = 7
IMG_SIZE    = 224
VAL_SPLIT   = 0.2
BATCH_SIZE  = 64

DATA_ROOT = "./ham10000"
CSV_PATH  = os.path.join(DATA_ROOT, "HAM10000_metadata.csv")
IMG_DIR   = os.path.join(DATA_ROOT, "images")

# Path to the weights saved by the BranchyNet training script
SAVE_PATH = "__ham10000__branchynet_resnet50.pth"

# Exit confidence thresholds to sweep during adaptive evaluation
EXIT_THRESHOLDS = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]

HAM10000_CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Using device: {DEVICE}")

# ── Dataset (same class used at training time) ──────────────────
class HAM10000Dataset(Dataset):
    def __init__(self, df, img_dir, transform=None, class_list=HAM10000_CLASSES):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform
        self.class2idx = {cls: i for i, cls in enumerate(class_list)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row["image_id"] + ".jpg")
        image    = Image.open(img_path).convert("RGB")
        label    = self.class2idx[row["dx"]]
        if self.transform:
            image = self.transform(image)
        return image, label

# ── AUXILIARY BRANCH (must match training architecture exactly) ──
class EarlyExitBranch(nn.Module):
    def __init__(self, input_channels: int, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.branch = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.BatchNorm1d(input_channels),
            nn.Linear(input_channels, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.branch(x)

# ── BRANCHYNET MODEL (must match training architecture exactly) ──
class BranchyResNet50(nn.Module):
    def __init__(self, num_classes: int = NUM_CLASSES, pretrained: bool = False):
        super().__init__()
        weights  = models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.resnet50(weights=weights)
        backbone.fc = nn.Linear(backbone.fc.in_features, num_classes)

        self.stem    = nn.Sequential(backbone.conv1, backbone.bn1,
                                     backbone.relu, backbone.maxpool)
        self.layer1  = backbone.layer1
        self.layer2  = backbone.layer2
        self.layer3  = backbone.layer3
        self.layer4  = backbone.layer4
        self.avgpool = backbone.avgpool
        self.fc      = backbone.fc

        self.branch1 = EarlyExitBranch(256, num_classes)
        self.branch2 = EarlyExitBranch(512, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        logits1 = self.branch1(x)
        x = self.layer2(x)
        logits2 = self.branch2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        logits3 = self.fc(x)
        return logits1, logits2, logits3

    @torch.no_grad()
    def adaptive_forward(self, x, threshold: float = 0.8):
        B = x.size(0)
        x = self.stem(x)

        x = self.layer1(x)
        logits1 = self.branch1(x)
        conf1   = torch.softmax(logits1, dim=1).max(dim=1).values
        if (conf1 >= threshold).all():
            exit_idx = torch.zeros(B, dtype=torch.long, device=x.device)
            return logits1, exit_idx

        x = self.layer2(x)
        logits2 = self.branch2(x)
        conf2   = torch.softmax(logits2, dim=1).max(dim=1).values
        if (conf2 >= threshold).all():
            exit_idx = torch.ones(B, dtype=torch.long, device=x.device)
            return logits2, exit_idx

        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        logits3 = self.fc(x)

        exit_idx = torch.full((B,), 2, dtype=torch.long, device=x.device)
        exit_idx[conf2 >= threshold] = 1
        exit_idx[conf1 >= threshold] = 0

        final_logits = logits3.clone()
        mask2 = conf2 >= threshold
        final_logits[mask2] = logits2[mask2]
        mask1 = conf1 >= threshold
        final_logits[mask1] = logits1[mask1]

        return final_logits, exit_idx

# ── Load the saved BranchyNet weights ────────────────────────────
model = BranchyResNet50(num_classes=NUM_CLASSES, pretrained=False)
model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
model = model.to(DEVICE).eval()
total_params = sum(p.numel() for p in model.parameters())
print(f"✓ Weights loaded from {SAVE_PATH}")
print(f"Total parameters (BranchyNet): {total_params:,}")

# ── Rebuild the SAME validation split used at training time ─────
df = pd.read_csv(CSV_PATH)
df = df[["image_id", "dx"]].drop_duplicates(subset="image_id").reset_index(drop=True)

_, val_df = train_test_split(
    df, test_size=VAL_SPLIT, random_state=SEED, stratify=df["dx"]
)
print(f"Validation set: {len(val_df)} images")

transform_val = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_set    = HAM10000Dataset(val_df, IMG_DIR, transform=transform_val)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# ── FULL EVALUATION (final exit only) ────────────────────────────
print("\n" + "="*60)
print("FULL EVALUATION — final exit (no early stopping)")
print("="*60)

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(DEVICE)
        _, _, logits3 = model(inputs)
        all_preds.extend(logits3.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc       = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average="macro", zero_division=0)
recall    = recall_score(all_labels, all_preds, average="macro", zero_division=0)
f1        = f1_score(all_labels, all_preds, average="macro", zero_division=0)

print(f"  Accuracy          : {acc:.4f}")
print(f"  Precision (macro) : {precision:.4f}")
print(f"  Recall    (macro) : {recall:.4f}")
print(f"  F1-score  (macro) : {f1:.4f}")

# ── ADAPTIVE COMPUTATION EVALUATION (threshold sweep) ────────────
print("\n" + "="*60)
print("ADAPTIVE COMPUTATION — Early-Exit Threshold Sweep")
print("="*60)
print(f"  Thresholds tested: {EXIT_THRESHOLDS}")

adaptive_results = []
for tau in EXIT_THRESHOLDS:
    preds_list, labels_list, exit_idx_list = [], [], []

    t_start = time.time()
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(DEVICE)
            logits, exit_idx = model.adaptive_forward(inputs, threshold=tau)
            preds_list.extend(logits.argmax(1).cpu().numpy())
            labels_list.extend(labels.numpy())
            exit_idx_list.extend(exit_idx.cpu().numpy())
    t_end = time.time()

    preds_arr    = np.array(preds_list)
    labels_arr   = np.array(labels_list)
    exit_idx_arr = np.array(exit_idx_list)
    n            = len(labels_arr)

    acc_t  = accuracy_score(labels_arr, preds_arr)
    prec_t = precision_score(labels_arr, preds_arr, average="macro", zero_division=0)
    rec_t  = recall_score(labels_arr, preds_arr, average="macro", zero_division=0)
    f1_t   = f1_score(labels_arr, preds_arr, average="macro", zero_division=0)

    frac_exit1  = (exit_idx_arr == 0).mean()
    frac_exit2  = (exit_idx_arr == 1).mean()
    frac_exit3  = (exit_idx_arr == 2).mean()
    avg_time_ms = (t_end - t_start) / n * 1000

    adaptive_results.append({
        "threshold"  : tau,
        "accuracy"   : round(acc_t,  6),
        "precision"  : round(prec_t, 6),
        "recall"     : round(rec_t,  6),
        "f1"         : round(f1_t,   6),
        "frac_exit1" : round(frac_exit1, 4),
        "frac_exit2" : round(frac_exit2, 4),
        "frac_exit3" : round(frac_exit3, 4),
        "avg_time_ms": round(avg_time_ms, 4),
    })

    print(f"  τ={tau:.2f} | Acc={acc_t:.4f} | F1={f1_t:.4f} | "
          f"Exit1={frac_exit1:.1%} Exit2={frac_exit2:.1%} Exit3={frac_exit3:.1%} | "
          f"Time={avg_time_ms:.4f}ms/sample")

best_adaptive = max(adaptive_results, key=lambda r: r["accuracy"])

# ── FLOPs ─────────────────────────────────────────────────────
dummy_flops = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
macs, _     = thop_profile(model, inputs=(dummy_flops,), verbose=False)
flops_G     = macs * 2 / 1e9

# ── Parameters & disk size ───────────────────────────────────────
size_mb = os.path.getsize(SAVE_PATH) / 1e6

# # ── Inference time — CPU (full model, all 3 exits) ──────────────
# print("\n⏱  Measuring CPU inference times ...")
# model_cpu = BranchyResNet50(num_classes=NUM_CLASSES, pretrained=False)
# model_cpu.load_state_dict(torch.load(SAVE_PATH, map_location="cpu"))
# model_cpu = model_cpu.eval()

# dummy_single_cpu = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
# dummy_batch_cpu  = torch.randn(128, 3, IMG_SIZE, IMG_SIZE)

# with torch.no_grad():
#     for _ in range(10):
#         model_cpu(dummy_single_cpu)

# times_cpu_single = []
# with torch.no_grad():
#     for _ in range(100):
#         t0 = time.perf_counter()
#         model_cpu(dummy_single_cpu)
#         times_cpu_single.append(time.perf_counter() - t0)
# inf_ms_single_cpu = np.mean(times_cpu_single) * 1000

# with torch.no_grad():
#     for _ in range(5):
#         model_cpu(dummy_batch_cpu)

# times_cpu_batch = []
# with torch.no_grad():
#     for _ in range(20):
#         t0 = time.perf_counter()
#         model_cpu(dummy_batch_cpu)
#         times_cpu_batch.append(time.perf_counter() - t0)
# inf_ms_batch128_cpu     = np.mean(times_cpu_batch) * 1000
# inf_ms_per_img_cpu      = inf_ms_batch128_cpu / 128
# throughput_imgs_sec_cpu = 128 / (inf_ms_batch128_cpu / 1000)

# del model_cpu, dummy_single_cpu, dummy_batch_cpu
# print("✓ CPU timing done")

# ── Inference time — GPU (full model, all 3 exits) ──────────────
for i in range(10):
    use_cuda = DEVICE.type == "cuda"
    print("⏱  Measuring GPU inference times ..." if use_cuda
        else "⏱  No CUDA device found — measuring 'GPU' timings on CPU instead ...")

    dummy_single_gpu = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

    with torch.no_grad():
        for _ in range(50):
            model(dummy_single_gpu)
    if use_cuda:
        torch.cuda.synchronize()

    times_gpu_single = []
    with torch.no_grad():
        for _ in range(500):
            if use_cuda:
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(dummy_single_gpu)
            if use_cuda:
                torch.cuda.synchronize()
            times_gpu_single.append(time.perf_counter() - t0)
    inf_ms_single_gpu = np.mean(times_gpu_single) * 1000

    dummy_batch_gpu = torch.randn(128, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

    if use_cuda:
        start_ev = torch.cuda.Event(enable_timing=True)
        end_ev   = torch.cuda.Event(enable_timing=True)
        with torch.no_grad():
            for _ in range(10):
                model(dummy_batch_gpu)
        torch.cuda.synchronize()

        batch_times_gpu = []
        with torch.no_grad():
            for _ in range(100):
                start_ev.record()
                model(dummy_batch_gpu)
                end_ev.record()
                torch.cuda.synchronize()
                batch_times_gpu.append(start_ev.elapsed_time(end_ev))
        inf_ms_batch128_gpu = np.mean(batch_times_gpu)
        gpu_timing_method = "CUDA events + torch.cuda.synchronize()"
    else:
        with torch.no_grad():
            for _ in range(5):
                model(dummy_batch_gpu)
        batch_times_gpu = []
        with torch.no_grad():
            for _ in range(20):
                t0 = time.perf_counter()
                model(dummy_batch_gpu)
                batch_times_gpu.append((time.perf_counter() - t0) * 1000)
        inf_ms_batch128_gpu = np.mean(batch_times_gpu)
        gpu_timing_method = "time.perf_counter() (no CUDA available)"

    inf_ms_per_img_gpu      = inf_ms_batch128_gpu / 128
    throughput_imgs_sec_gpu = 128 / (inf_ms_batch128_gpu / 1000)

    print(f"✓ GPU timing done for {i}")
    print(f"  --- Inference (GPU) ---")
    print(f"  Single image GPU  : {inf_ms_single_gpu:.3f} ms")
    print(f"  Batch-128 GPU     : {inf_ms_batch128_gpu:.2f} ms")
    print(f"  Per-image GPU     : {inf_ms_per_img_gpu:.3f} ms")
    print(f"  Throughput GPU    : {throughput_imgs_sec_gpu:.1f} imgs/sec")
    print("\n")
print("✓ Loop for GPU timing is done")


# ── Adaptive inference timing (τ=0.8, single image) ──────────────
with torch.no_grad():
    for _ in range(3):
        model.adaptive_forward(dummy_single_gpu, threshold=0.8)
    if use_cuda:
        torch.cuda.synchronize()
        start_evt = torch.cuda.Event(enable_timing=True)
        end_evt   = torch.cuda.Event(enable_timing=True)
        start_evt.record()
        for _ in range(50):
            model.adaptive_forward(dummy_single_gpu, threshold=0.8)
        end_evt.record()
        torch.cuda.synchronize()
        inf_ms_adapt_tau08 = start_evt.elapsed_time(end_evt) / 50
    else:
        t0 = time.perf_counter()
        for _ in range(50):
            model.adaptive_forward(dummy_single_gpu, threshold=0.8)
        inf_ms_adapt_tau08 = (time.perf_counter() - t0) / 50 * 1000

# ── Print results ─────────────────────────────────────────────
print(f"\n{'='*60}")
print("BRANCHYNET METRICS (recomputed from saved weights)")
print(f"{'='*60}")
print(f"  Accuracy (final exit) : {acc:.4f}")
print(f"  Precision (macro)     : {precision:.4f}")
print(f"  Recall    (macro)     : {recall:.4f}")
print(f"  F1-score  (macro)     : {f1:.4f}")
print(f"  Parameters            : {total_params:,}")
print(f"  Model size            : {size_mb:.2f} MB")
print(f"  FLOPs (full, all exits): {flops_G:.3f} GFLOPs")
# print(f"  --- Inference (CPU, full model) ---")
# print(f"  Single image      : {inf_ms_single_cpu:.3f} ms")
# print(f"  Batch-128         : {inf_ms_batch128_cpu:.2f} ms")
# print(f"  Throughput        : {throughput_imgs_sec_cpu:.1f} imgs/sec")
print(f"  --- Inference (GPU, full model) ---")
print(f"  Single image      : {inf_ms_single_gpu:.3f} ms")
print(f"  Batch-128         : {inf_ms_batch128_gpu:.2f} ms")
print(f"  Throughput        : {throughput_imgs_sec_gpu:.1f} imgs/sec")
print(f"  --- Adaptive (τ=0.8, single image) ---")
print(f"  Latency           : {inf_ms_adapt_tau08:.3f} ms")
print(f"\n  Best adaptive result (τ={best_adaptive['threshold']}):")
print(f"    Accuracy   : {best_adaptive['accuracy']:.4f}")
print(f"    Exit1      : {best_adaptive['frac_exit1']:.1%}")
print(f"    Exit2      : {best_adaptive['frac_exit2']:.1%}")
print(f"    Exit3      : {best_adaptive['frac_exit3']:.1%}")

# ── Save metrics JSON (same schema as the training script) ──────
branchynet_metrics = {
    "accuracy"  : acc,
    "precision" : precision,
    "recall"    : recall,
    "f1"        : f1,
    "params"    : total_params,
    "size_mb"   : size_mb,
    "flops_G"   : flops_G,
    "inference_ms": {
        # "cpu": {
        #     "single_img_ms"      : round(inf_ms_single_cpu, 4),
        #     "batch128_ms"        : round(inf_ms_batch128_cpu, 4),
        #     "per_img_ms"         : round(inf_ms_per_img_cpu, 4),
        #     "throughput_imgs_sec": round(throughput_imgs_sec_cpu, 1),
        #     "timing_method"      : "time.perf_counter()",
        # },
        "gpu": {
            "single_img_ms"      : round(inf_ms_single_gpu, 4),
            "batch128_ms"        : round(inf_ms_batch128_gpu, 4),
            "per_img_ms"         : round(inf_ms_per_img_gpu, 4),
            "throughput_imgs_sec": round(throughput_imgs_sec_gpu, 1),
            "timing_method"      : gpu_timing_method,
        },
    },
    "method"                : "early_exit",
    "variant"               : "BranchyNet_ResNet50",
    "dataset"                : "HAM10000",
    "num_exits"              : 3,
    "exit_positions"         : ["after layer1 (256ch)", "after layer2 (512ch)", "final fc (2048ch)"],
    "exit_thresholds_tested" : EXIT_THRESHOLDS,
    "inference_ms_adaptive_tau08_single": round(inf_ms_adapt_tau08, 4),
    "adaptive_threshold_results"        : adaptive_results,
    "best_adaptive_result"              : best_adaptive,
    "saved_model_path": os.path.abspath(SAVE_PATH),
}

with open("__ham10000__branchynet_metrics_v2.json", "w") as f:
    json.dump(branchynet_metrics, f, indent=2)
print(f"\n✓ Metrics saved → __ham10000__branchynet_metrics_v2.json")

Using device: cuda
✓ Weights loaded from __ham10000__branchynet_resnet50.pth
Total parameters (BranchyNet): 23,624,277
Validation set: 2003 images

FULL EVALUATION — final exit (no early stopping)
  Accuracy          : 0.8882
  Precision (macro) : 0.8389
  Recall    (macro) : 0.8454
  F1-score  (macro) : 0.8403

ADAPTIVE COMPUTATION — Early-Exit Threshold Sweep
  Thresholds tested: [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
  τ=0.50 | Acc=0.8013 | F1=0.6483 | Exit1=65.5% Exit2=18.0% Exit3=16.5% | Time=8.1371ms/sample
  τ=0.60 | Acc=0.8352 | F1=0.7044 | Exit1=54.4% Exit2=16.7% Exit3=28.9% | Time=8.6320ms/sample
  τ=0.70 | Acc=0.8597 | F1=0.7554 | Exit1=45.4% Exit2=13.2% Exit3=41.3% | Time=8.6352ms/sample
  τ=0.80 | Acc=0.8817 | F1=0.8122 | Exit1=36.1% Exit2=11.5% Exit3=52.4% | Time=8.6355ms/sample
  τ=0.90 | Acc=0.8857 | F1=0.8318 | Exit1=27.5% Exit2=10.1% Exit3=62.4% | Time=8.6423ms/sample
  τ=0.95 | Acc=0.8882 | F1=0.8374 | Exit1=18.2% Exit2=10.6% Exit3=71.2% | Time=7.0459ms/sample
⏱  Measuring 